# 📚 Silver Layer - Data Governance & Documentation

## 🎯 Objective
This notebook applies **Unity Catalog best practices** to document and govern the Silver layer tables in the `retail_dev.silver` catalog. Proper governance ensures data discoverability, compliance, and trust across the organization.

## 📊 Scope
We will apply governance to **7 Silver tables**:
1. **customers** - SCD Type 2 dimension with PII
2. **suppliers** - Supplier master dimension
3. **products** - Product catalog dimension
4. **loyalty_segments** - Loyalty program dimension
5. **sales_orders** - Fact table (aggregated, one row per order)
6. **sales_order_items** - Fact detail table (one row per item)
7. **sales_order_clicks** - Event table (one row per click)

## 🛡️ Best Practices Covered
* **COMMENT ON** - Add descriptive comments to tables and columns
* **PII Tags** - Mark sensitive data (customer names, tax IDs)
* **Table Properties** - Owner, classification, update frequency, SCD type
* **Documentation** - Business definitions and grain specifications

---

In [0]:
# Configuration
CATALOG = "retail_dev"
SCHEMA = "silver"

print(f"🔧 Configuration:")
print(f"   Catalog: {CATALOG}")
print(f"   Schema: {SCHEMA}")
print(f"\n✅ Ready to apply governance!")

## 🧑‍🤝‍🧑 Table: customers

**Type**: Dimension (SCD Type 2)  
**Grain**: One row per customer per validity period  
**PII**: Yes (customer_name, tax_id)  
**Update Frequency**: Daily via Bronze layer

In [0]:
# Add table-level comment for customers
table_name = f"{CATALOG}.{SCHEMA}.customers"

table_comment = """
Customer master dimension table implementing SCD Type 2 for historical tracking. 
Contains customer demographics, contact information, and address details. 
Updated daily via bronze layer with temporal validity tracking (valid_from, valid_to, is_current).
""".strip()

spark.sql(f"COMMENT ON TABLE {table_name} IS '{table_comment}'")
print(f"✅ Table comment added to {table_name}")

In [0]:
# Add column comments for customers
table_name = f"{CATALOG}.{SCHEMA}.customers"

column_comments = {
    'CUSTOMER_ID': 'Unique customer identifier (Primary Key)',
    'CUSTOMER_NAME': 'Full customer name (PII - Personal Identifiable Information)',
    'TAX_ID': 'Tax identification number (PII - Highly Sensitive)',
    'STATE': 'State or province of customer address',
    'CITY': 'City of customer address',
    'POSTCODE': 'Postal code, cleaned and standardized (removed trailing .0)',
    'STREET': 'Street name of customer address',
    'NUMBER': 'Street number of customer address',
    'UNIT': 'Unit or apartment number',
    'REGION': 'Geographic region classification',
    'DISTRICT': 'District or neighborhood',
    'LON': 'Longitude coordinate (validated range: -180 to 180)',
    'LAT': 'Latitude coordinate (validated range: -90 to 90)',
    'VALID_FROM': 'SCD Type 2: Record validity start timestamp',
    'VALID_TO': 'SCD Type 2: Record validity end timestamp (NULL = current)',
    'IS_CURRENT': 'SCD Type 2: Flag indicating current/active record (1=current, 0=historical)',
    'DATA_QUALITY_SCORE': 'Completeness score: percentage of non-null fields (0-100%)',
    'PROCESSED_AT': 'Timestamp when record was processed into Silver layer',
    'SOURCE_TABLE': 'Source Bronze table name',
    'PIPELINE_RUN_ID': 'Unique identifier for pipeline execution run'
}

for column, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN {column} COMMENT '{comment}'")

print(f"✅ Column comments added to {table_name} ({len(column_comments)} columns)")

In [0]:
# Apply PII tags to sensitive columns
table_name = f"{CATALOG}.{SCHEMA}.customers"

# Tag CUSTOMER_NAME as PII
spark.sql(f"""
ALTER TABLE {table_name} 
ALTER COLUMN CUSTOMER_NAME 
SET TAGS ('PII'='true', 'sensitivity'='medium')
""")
print(f"✅ PII tag applied to CUSTOMER_NAME")

# Tag TAX_ID as PII with high sensitivity
spark.sql(f"""
ALTER TABLE {table_name} 
ALTER COLUMN TAX_ID 
SET TAGS ('PII'='true', 'sensitivity'='high')
""")
print(f"✅ PII tag applied to TAX_ID (high sensitivity)")

In [0]:
# Set table properties for customers
table_name = f"{CATALOG}.{SCHEMA}.customers"

spark.sql(f"""
ALTER TABLE {table_name} 
SET TBLPROPERTIES (
  'data_classification' = 'silver',
  'layer' = 'silver',
  'update_frequency' = 'daily',
  'scd_type' = 'type2',
  'contains_pii' = 'true',
  'quality_check_enabled' = 'true'
)
""")

print(f"✅ Table properties set for {table_name}")

## 🏭 Table: suppliers

**Type**: Dimension  
**Grain**: One row per unique supplier  
**PII**: Partial (supplier_name - business information)  
**Update Frequency**: Daily via Bronze layer

In [0]:
# Add table-level comment for suppliers
table_name = f"{CATALOG}.{SCHEMA}.suppliers"

table_comment = """
Supplier master dimension table containing supplier business information and locations. 
Duplicates removed based on SUPPLIER_ID. Updated daily via bronze layer.
""".strip()

spark.sql(f"COMMENT ON TABLE {table_name} IS '{table_comment}'")
print(f"✅ Table comment added to {table_name}")

In [0]:
# Add column comments for suppliers
table_name = f"{CATALOG}.{SCHEMA}.suppliers"

column_comments = {
    'SUPPLIER_ID': 'Unique supplier identifier (Primary Key)',
    'TAX_ID': 'Business tax identification number',
    'SUPPLIER_NAME': 'Supplier business name (potentially sensitive)',
    'STATE': 'State or province of supplier location',
    'CITY': 'City of supplier location',
    'POSTCODE': 'Postal code, cleaned and standardized (converted from DOUBLE)',
    'STREET': 'Street name of supplier address',
    'NUMBER': 'Street number of supplier address',
    'UNIT': 'Unit or building number',
    'REGION': 'Geographic region classification',
    'DISTRICT': 'District or area',
    'LON': 'Longitude coordinate (validated range: -180 to 180)',
    'LAT': 'Latitude coordinate (validated range: -90 to 90)',
    'ITEMS_PROVIDED': 'List or description of items supplied',
    'DATA_QUALITY_SCORE': 'Completeness score (0-100%)',
    'PROCESSED_AT': 'Processing timestamp',
    'SOURCE_TABLE': 'Source Bronze table',
    'PIPELINE_RUN_ID': 'Pipeline execution identifier'
}

for column, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN {column} COMMENT '{comment}'")

print(f"✅ Column comments added to {table_name} ({len(column_comments)} columns)")

In [0]:
# Apply tags to supplier business information
table_name = f"{CATALOG}.{SCHEMA}.suppliers"

# Tag SUPPLIER_NAME (business name can be considered sensitive)
spark.sql(f"""
ALTER TABLE {table_name} 
ALTER COLUMN SUPPLIER_NAME 
SET TAGS ('business_info'='true', 'sensitivity'='low')
""")
print(f"✅ Business info tag applied to SUPPLIER_NAME")

In [0]:
# Set table properties for suppliers
table_name = f"{CATALOG}.{SCHEMA}.suppliers"

spark.sql(f"""
ALTER TABLE {table_name} 
SET TBLPROPERTIES (
  'data_classification' = 'silver',
  'layer' = 'silver',
  'update_frequency' = 'daily',
  'contains_pii' = 'false'
)
""")

print(f"✅ Table properties set for {table_name}")

## 📦 Table: products

**Type**: Dimension  
**Grain**: One row per unique product  
**PII**: None  
**Update Frequency**: Daily via Bronze layer

In [0]:
# Add table-level comment for products
table_name = f"{CATALOG}.{SCHEMA}.products"

table_comment = """
Product catalog dimension table containing product master data including names, categories, 
pricing, and barcodes (EAN13, EAN5). Updated daily via bronze layer.
""".strip()

spark.sql(f"COMMENT ON TABLE {table_name} IS '{table_comment}'")
print(f"✅ Table comment added to {table_name}")

In [0]:
# Add column comments for products
table_name = f"{CATALOG}.{SCHEMA}.products"

column_comments = {
    'PRODUCT_ID': 'Unique product identifier (Primary Key)',
    'PRODUCT_CATEGORY': 'Product category classification',
    'PRODUCT_NAME': 'Product display name',
    'SALES_PRICE': 'Current sales price per unit',
    'EAN13': 'European Article Number (13-digit barcode)',
    'EAN5': 'European Article Number (5-digit supplemental barcode)',
    'PRODUCT_UNIT': 'Unit of measure (e.g., kg, liter, piece)',
    'DATA_QUALITY_SCORE': 'Completeness score (0-100%)',
    'PROCESSED_AT': 'Processing timestamp',
    'SOURCE_TABLE': 'Source Bronze table',
    'PIPELINE_RUN_ID': 'Pipeline execution identifier'
}

for column, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN {column} COMMENT '{comment}'")

print(f"✅ Column comments added to {table_name} ({len(column_comments)} columns)")

In [0]:
# Set table properties for products
table_name = f"{CATALOG}.{SCHEMA}.products"

spark.sql(f"""
ALTER TABLE {table_name} 
SET TBLPROPERTIES (
  'data_classification' = 'silver',
  'layer' = 'silver',
  'update_frequency' = 'daily',
  'contains_pii' = 'false'
)
""")

print(f"✅ Table properties set for {table_name}")
print(f"ℹ️  No PII tags needed for products table")

## 💎 Table: loyalty_segments

**Type**: Dimension (with temporal validity)  
**Grain**: One row per loyalty segment per validity period  
**PII**: None  
**Update Frequency**: As needed (segment definitions change infrequently)

In [0]:
# Add table-level comment for loyalty_segments
table_name = f"{CATALOG}.{SCHEMA}.loyalty_segments"

table_comment = """
Loyalty program dimension table defining customer loyalty tiers and thresholds. 
Contains temporal validity (valid_from, valid_to) to track changes in segment definitions over time. 
Used to classify customers based on purchase behavior and unit thresholds.
""".strip()

spark.sql(f"COMMENT ON TABLE {table_name} IS '{table_comment}'")
print(f"✅ Table comment added to {table_name}")

In [0]:
# Add column comments for loyalty_segments
table_name = f"{CATALOG}.{SCHEMA}.loyalty_segments"

column_comments = {
    'LOYALTY_SEGMENT_ID': 'Unique loyalty segment identifier (Primary Key)',
    'LOYALTY_SEGMENT_DESCRIPTION': 'Business description of the loyalty tier (e.g., Bronze, Silver, Gold)',
    'UNIT_THRESHOLD': 'Minimum units required to qualify for this segment',
    'VALID_FROM': 'Segment definition validity start date',
    'VALID_TO': 'Segment definition validity end date (NULL = currently active)',
    'IS_CURRENT': 'Flag indicating current/active segment definition (1=current, 0=historical)',
    'DATA_QUALITY_SCORE': 'Completeness score (0-100%)',
    'PROCESSED_AT': 'Processing timestamp',
    'SOURCE_TABLE': 'Source Bronze table',
    'PIPELINE_RUN_ID': 'Pipeline execution identifier'
}

for column, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN {column} COMMENT '{comment}'")

print(f"✅ Column comments added to {table_name} ({len(column_comments)} columns)")

In [0]:
# Set table properties for loyalty_segments
table_name = f"{CATALOG}.{SCHEMA}.loyalty_segments"

spark.sql(f"""
ALTER TABLE {table_name} 
SET TBLPROPERTIES (
  'data_classification' = 'silver',
  'layer' = 'silver',
  'update_frequency' = 'as_needed',
  'temporal_tracking' = 'true',
  'contains_pii' = 'false'
)
""")

print(f"✅ Table properties set for {table_name}")
print(f"ℹ️  No PII tags needed for loyalty segments table")

## 🛒 Table: sales_orders (Fact - Aggregated)

**Type**: Fact Table (aggregated/header level)  
**Grain**: One row per order  
**PII**: None (order data only)  
**Update Frequency**: Daily via Bronze layer  
**Special**: Contains nested arrays (clicked_items, ordered_products, promo_info)

In [0]:
# Add table-level comment for sales_orders
table_name = f"{CATALOG}.{SCHEMA}.sales_orders"

table_comment = """
Sales order fact table at header/aggregated level - one row per order. 
Contains order-level attributes including customer reference, order timestamp, and nested arrays 
for clicked items, ordered products, and promotion information. This is an append-only fact table 
(no SCD Type 2). For detailed line-item analysis, use sales_order_items. 
For click behavior analysis, use sales_order_clicks.
""".strip()

spark.sql(f"COMMENT ON TABLE {table_name} IS '{table_comment}'")
print(f"✅ Table comment added to {table_name}")

In [0]:
# Add column comments for sales_orders
table_name = f"{CATALOG}.{SCHEMA}.sales_orders"

column_comments = {
    'ORDER_NUMBER': 'Unique order identifier (Primary Key)',
    'CUSTOMER_ID': 'Foreign key to customers dimension',
    'CUSTOMER_NAME': 'Customer name at time of order (denormalized for convenience)',
    'ORDER_DATETIME': 'Order timestamp (converted from Unix epoch to TIMESTAMP)',
    'NUMBER_OF_LINE_ITEMS': 'Total count of distinct products in the order',
    'CLICKED_ITEMS': 'Nested array of products clicked during shopping session: ARRAY<ARRAY<STRING>> where each element is [product_id, click_score]',
    'ORDERED_PRODUCTS': 'Nested array of purchased products: ARRAY<STRUCT<curr, id, name, price, qty, unit, promotion_info>>',
    'PROMO_INFO': 'Nested array of order-level promotions applied: ARRAY<STRUCT<promo_disc, promo_id, promo_item, promo_qty>>',
    'DATA_QUALITY_SCORE': 'Completeness score (0-100%)',
    'PROCESSED_AT': 'Processing timestamp',
    'SOURCE_TABLE': 'Source Bronze table',
    'PIPELINE_RUN_ID': 'Pipeline execution identifier'
}

for column, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN {column} COMMENT '{comment}'")

print(f"✅ Column comments added to {table_name} ({len(column_comments)} columns)")

In [0]:
# Set table properties for sales_orders
table_name = f"{CATALOG}.{SCHEMA}.sales_orders"

spark.sql(f"""
ALTER TABLE {table_name} 
SET TBLPROPERTIES (
  'data_classification' = 'silver',
  'layer' = 'silver',
  'update_frequency' = 'daily',
  'fact_type' = 'transactional',
  'grain' = 'one_row_per_order',
  'contains_pii' = 'false',
  'contains_nested_data' = 'true',
  'related_detail_tables' = 'sales_order_items, sales_order_clicks'
)
""")

print(f"✅ Table properties set for {table_name}")
print(f"ℹ️  No PII tags needed - fact table with transactional data only")

## 📋 Table: sales_order_items (Fact Detail)

**Type**: Fact Table (detail/line-item level)  
**Grain**: One row per order line item (per product ordered)  
**PII**: None  
**Parent**: sales_orders  
**Update Frequency**: Daily via Bronze layer

In [0]:
# Add table-level comment for sales_order_items
table_name = f"{CATALOG}.{SCHEMA}.sales_order_items"

table_comment = """
Sales order line-item fact table created by exploding the ORDERED_PRODUCTS array from sales_orders. 
Each row represents one product within an order. Contains flattened product details including pricing, 
quantity, and item-level promotion information. Use this table for product-level sales analysis, 
revenue calculations, and promotion effectiveness studies.
""".strip()

spark.sql(f"COMMENT ON TABLE {table_name} IS '{table_comment}'")
print(f"✅ Table comment added to {table_name}")

In [0]:
# Add column comments for sales_order_items
table_name = f"{CATALOG}.{SCHEMA}.sales_order_items"

column_comments = {
    'ORDER_ID': 'Foreign key to sales_orders (part of composite key)',
    'CUSTOMER_ID': 'Foreign key to customers dimension',
    'ORDER_DATETIME': 'Order timestamp (from parent sales_orders)',
    'ITEM_POSITION': 'Sequential position of item within the order (1-based)',
    'PRODUCT_ID': 'Foreign key to products dimension',
    'PRODUCT_NAME': 'Product name at time of order',
    'CURRENCY': 'Currency code for pricing',
    'PRICE': 'Unit price of the product',
    'QUANTITY': 'Quantity ordered',
    'UNIT': 'Unit of measure',
    'ITEM_TOTAL': 'Calculated total: PRICE × QUANTITY',
    'PROMO_DISC': 'Item-level promotion discount amount',
    'PROMO_ID': 'Item-level promotion identifier',
    'PROMO_ITEM': 'Item-level promotion description',
    'PROMO_QTY': 'Quantity eligible for promotion',
    'HAS_ITEM_PROMO': 'Boolean flag: 1 if item has promotion, 0 otherwise',
    'PROCESSED_AT': 'Processing timestamp',
    'SOURCE_TABLE': 'Source Silver table (sales_orders)',
    'PIPELINE_RUN_ID': 'Pipeline execution identifier'
}

for column, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN {column} COMMENT '{comment}'")

print(f"✅ Column comments added to {table_name} ({len(column_comments)} columns)")

In [0]:
# Set table properties for sales_order_items
table_name = f"{CATALOG}.{SCHEMA}.sales_order_items"

spark.sql(f"""
ALTER TABLE {table_name} 
SET TBLPROPERTIES (
  'data_classification' = 'silver',
  'layer' = 'silver',
  'update_frequency' = 'daily',
  'fact_type' = 'transactional',
  'grain' = 'one_row_per_item',
  'parent_table' = 'sales_orders',
  'contains_pii' = 'false',
  'derived_from' = 'ORDERED_PRODUCTS array explosion'
)
""")

print(f"✅ Table properties set for {table_name}")
print(f"ℹ️  No PII tags needed - derived fact table with product-level transactional data")

## 🖱️ Table: sales_order_clicks (Event Table)

**Type**: Event Fact Table  
**Grain**: One row per product click event  
**PII**: None  
**Parent**: sales_orders  
**Update Frequency**: Daily via Bronze layer

In [0]:
# Add table-level comment for sales_order_clicks
table_name = f"{CATALOG}.{SCHEMA}.sales_order_clicks"

table_comment = """
Click event fact table created by exploding the CLICKED_ITEMS array from sales_orders. 
Each row represents one product click during the shopping session. Contains click sequence 
(CLICK_POSITION) and click score for behavioral analysis. Use this table for conversion funnel 
analysis, product affinity studies, and cart abandonment investigations.
""".strip()

spark.sql(f"COMMENT ON TABLE {table_name} IS '{table_comment}'")
print(f"✅ Table comment added to {table_name}")

In [0]:
# Add column comments for sales_order_clicks
table_name = f"{CATALOG}.{SCHEMA}.sales_order_clicks"

column_comments = {
    'ORDER_ID': 'Foreign key to sales_orders (part of composite key)',
    'CUSTOMER_ID': 'Foreign key to customers dimension',
    'ORDER_DATETIME': 'Order timestamp (from parent sales_orders)',
    'CLICK_POSITION': 'Sequential position of click within shopping session (1-based)',
    'CLICKED_PRODUCT_ID': 'Product identifier that was clicked (may or may not be purchased)',
    'CLICK_SCORE': 'Click engagement score or weight',
    'PROCESSED_AT': 'Processing timestamp',
    'SOURCE_TABLE': 'Source Silver table (sales_orders)',
    'PIPELINE_RUN_ID': 'Pipeline execution identifier'
}

for column, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN {column} COMMENT '{comment}'")

print(f"✅ Column comments added to {table_name} ({len(column_comments)} columns)")

In [0]:
# Set table properties for sales_order_clicks
table_name = f"{CATALOG}.{SCHEMA}.sales_order_clicks"

spark.sql(f"""
ALTER TABLE {table_name} 
SET TBLPROPERTIES (
  'data_classification' = 'silver',
  'layer' = 'silver',
  'update_frequency' = 'daily',
  'fact_type' = 'behavioral',
  'grain' = 'one_row_per_click',
  'parent_table' = 'sales_orders',
  'contains_pii' = 'false',
  'derived_from' = 'CLICKED_ITEMS array explosion'
)
""")

print(f"✅ Table properties set for {table_name}")
print(f"ℹ️  No PII tags needed - behavioral event data only")

## ✅ Governance Summary

Let's verify all governance metadata has been applied correctly by querying table properties, comments, and tags.

In [0]:
# Query governance metadata for all Silver tables
import pandas as pd

tables = [
    "customers",
    "suppliers", 
    "products",
    "loyalty_segments",
    "sales_orders",
    "sales_order_items",
    "sales_order_clicks"
]

print("="*80)
print("📊 SILVER LAYER GOVERNANCE SUMMARY")
print("="*80)

for table in tables:
    table_name = f"{CATALOG}.{SCHEMA}.{table}"
    print(f"\n{'='*80}")
    print(f"📋 Table: {table_name}")
    print(f"{'='*80}")
    
    # Get table details
    try:
        details = spark.sql(f"DESCRIBE EXTENDED {table_name}").toPandas()
        
        # Extract table comment
        comment_row = details[details['col_name'] == 'Comment']
        if not comment_row.empty:
            comment = comment_row['data_type'].values[0]
            print(f"\n📝 Table Comment:\n   {comment[:150]}..." if len(str(comment)) > 150 else f"\n📝 Table Comment:\n   {comment}")
        
        # Extract table properties
        props_row = details[details['col_name'] == 'Table Properties']
        if not props_row.empty:
            props = props_row['data_type'].values[0]
            print(f"\n🏷️  Table Properties:\n   {props[:200]}..." if len(str(props)) > 200 else f"\n🏷️  Table Properties:\n   {props}")
        
        # Count columns with comments
        col_details = spark.sql(f"DESCRIBE TABLE {table_name}").toPandas()
        col_count = len(col_details[col_details['col_name'] != ''])
        commented_cols = len(col_details[col_details['comment'].notna() & (col_details['comment'] != '')])
        
        print(f"\n📊 Columns: {col_count} total, {commented_cols} with comments")
        
    except Exception as e:
        print(f"   ⚠️  Error querying {table}: {str(e)}")

print(f"\n{'='*80}")
print("✅ Governance metadata applied to all 7 Silver tables!")
print(f"{'='*80}")

## 🎯 Next Steps & Recommendations

### 🔄 Ongoing Governance Activities
1. **Periodic Reviews** (Quarterly)
   * Review and update table/column comments as business definitions evolve
   * Validate PII tags remain accurate as data changes
   * Update table properties (ownership, frequency) as needed

2. **Data Quality Monitoring**
   * Set up alerts on DATA_QUALITY_SCORE thresholds
   * Monitor PROCESSED_AT timestamps for pipeline delays
   * Track PIPELINE_RUN_ID for failure patterns

3. **Access Audits**
   * Review who has SELECT permissions on PII-tagged columns
   * Implement column-level masking for sensitive fields
   * Set up audit logging for PII access

4. **Documentation Expansion**
   * Add sample queries as table properties
   * Document known data quality issues
   * Link to business owners and SLAs

### 📈 Gold Layer Preparation
With Silver layer fully documented, you're ready to:
* Create aggregated metrics tables (sales by product, customer lifetime value)
* Build star schema for BI tools (fact-dimension relationships already defined)
* Implement data quality checks using the governance metadata

---

**✅ Silver Layer Governance Complete!**  
All tables are now documented, tagged, and ready for analytics and Gold layer development.